# Assom-style refined reproduction (clean notebook)

Goal: reproduce the refined result from Assom where clustering separates more than the 2 coarse call groups.

Pipeline:
1. Load metadata (Annotations + FileInfo)
2. Dynamic sub-segmentation (vocalseg-style) inside each annotated segment
3. Mel features in batches -> UMAP (`n_neighbors=30`, `min_dist=0.3`, `metric='euclidean'`)
4. HDBSCAN (`min_cluster_size~2%`, `min_samples=20`, `epsilon=0.1`, `leaf`)
5. HP1 (permutation / associative syntax), HP2 (Wilcoxon), HP3 (Maximal Repeats + power-law)


## 1) Config


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

DATA_ROOT = '/Volumes/T7/data'     # set to None to use PROJECT_ROOT/data
DATA_DIR = Path(DATA_ROOT) if DATA_ROOT else (PROJECT_ROOT / 'data')
RAW_DIR = DATA_DIR / 'raw' / 'fruitbat'
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Runtime limits
MAX_FILES = None            # e.g. 3000 for test
MAX_SEGMENTS = None         # cap number of resulting sub-segments
WAV_LOAD_TIMEOUT_SEC = 90

# Batch sizes for memory-safe processing
BATCH_FIT_SEGMENTS = 20_000
BATCH_TRANSFORM_SEGMENTS = 25_000

# Segmentation (from Assom notebooks)
USE_DYNAMIC_SUBSEG = True
SEG_N_FFT = 1024
SEG_HOP_LENGTH_MS = 0.5
SEG_WIN_LENGTH_MS = 4
SEG_REF_LEVEL_DB = 20
SEG_PRE = 0.97
SEG_MIN_LEVEL_DB = -30
SEG_SILENCE_THRESHOLD = 0.1
SEG_MIN_SYLLABLE_LENGTH_S = 0.01
SEG_SPECTRAL_RANGE = [5000, 60000]
SEG_MIN_LEVEL_DB_FLOOR = 20

# Feature extraction
SR = 250_000
N_FFT = 1024
HOP_LENGTH = 512
N_MELS = 32
PAD_LEN = 128
FMIN, FMAX = 500, 120_000

# UMAP/HDBSCAN from Assom refined settings
UMAP_N_NEIGHBORS = 30
UMAP_MIN_DIST = 0.3
UMAP_METRIC = 'euclidean'

HDBSCAN_FRAC_CANDIDATES = [0.02, 0.015, 0.01, 0.005]
HDBSCAN_MIN_SAMPLES = 20
HDBSCAN_EPS = 0.1
HDBSCAN_METHOD = 'leaf'

EXCLUDE_CONTEXTS = {0, 11, 12}
MIN_SAMPLES_PER_CONTEXT = 30
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

CONTEXT_NAMES = {
    1: 'Separation', 2: 'Biting', 3: 'Feeding', 4: 'Fighting',
    5: 'Grooming', 6: 'Isolation', 7: 'Kissing', 8: 'Landing',
    9: 'Mating protest', 10: 'Threat-like',
}

print('RAW_DIR:', RAW_DIR)
print('MAX_FILES:', MAX_FILES or 'all', '| MAX_SEGMENTS:', MAX_SEGMENTS or 'no cap')
print('Dynamic sub-segmentation:', USE_DYNAMIC_SUBSEG)
print('UMAP:', {'n_neighbors': UMAP_N_NEIGHBORS, 'min_dist': UMAP_MIN_DIST, 'metric': UMAP_METRIC})


## 2) Load metadata


In [ ]:
annotations = pd.read_csv(RAW_DIR / 'Annotations.csv')
annotations = annotations.astype({
    'FileID': int,
    'Emitter': int,
    'Addressee': int,
    'Context': int,
    'Start sample': int,
    'End sample': int,
})
annotations = annotations[~annotations['Context'].isin(EXCLUDE_CONTEXTS)].copy()
print('Annotations:', len(annotations))

with open(RAW_DIR / 'FileInfo.csv', 'r') as f:
    max_cols = max(len(line.split(',')) for line in f)

file_info = pd.read_csv(RAW_DIR / 'FileInfo.csv', header=None, names=range(max_cols), low_memory=False)
file_info = file_info.iloc[1:][[0, 2, 3]].copy()
file_info.columns = ['FileID', 'FileName', 'FileFolder']
file_info['FileID'] = file_info['FileID'].astype(int)
file_info = file_info.set_index('FileID')

wav_root = RAW_DIR / 'zip_contents'
file_info['wav_path'] = file_info.apply(
    lambda r: str(wav_root / str(r['FileFolder']).strip() / str(r['FileName']).strip()), axis=1
)

print('FileInfo:', len(file_info), '| WAV root:', wav_root)
print('Contexts:', sorted(annotations['Context'].unique().tolist()))


## 3) Dynamic sub-segmentation + batched mel feature extraction


In [ ]:
import librosa
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeoutError
from tqdm.auto import tqdm

try:
    from vocalseg.dynamic_thresholding import dynamic_threshold_segmentation
    HAS_VOCALSEG = True
except Exception:
    HAS_VOCALSEG = False

print('vocalseg available:', HAS_VOCALSEG)
if USE_DYNAMIC_SUBSEG and not HAS_VOCALSEG:
    print('WARNING: dynamic sub-seg requested but vocalseg missing. Falling back to original segments.')


def _load_wav(path):
    return librosa.load(path, sr=SR, mono=True)


def make_mel(audio):
    S = librosa.feature.melspectrogram(
        y=audio,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmin=FMIN,
        fmax=FMAX,
    )
    S_db = librosa.power_to_db(S, ref=np.max)
    if S_db.shape[1] < PAD_LEN:
        S_db = np.pad(S_db, ((0, 0), (0, PAD_LEN - S_db.shape[1])), mode='constant', constant_values=S_db.min())
    else:
        S_db = S_db[:, :PAD_LEN]
    return S_db.ravel().astype(np.float32)


def dynamic_subsegments(seg_audio):
    if not (USE_DYNAMIC_SUBSEG and HAS_VOCALSEG):
        return [(0, len(seg_audio))]

    try:
        r = dynamic_threshold_segmentation(
            seg_audio,
            SR,
            n_fft=SEG_N_FFT,
            hop_length_ms=SEG_HOP_LENGTH_MS,
            win_length_ms=SEG_WIN_LENGTH_MS,
            ref_level_db=SEG_REF_LEVEL_DB,
            pre=SEG_PRE,
            min_level_db=SEG_MIN_LEVEL_DB,
            silence_threshold=SEG_SILENCE_THRESHOLD,
            spectral_range=SEG_SPECTRAL_RANGE,
            min_syllable_length_s=SEG_MIN_SYLLABLE_LENGTH_S,
            min_level_db_floor=SEG_MIN_LEVEL_DB_FLOOR,
            verbose=False,
        )
    except Exception:
        return [(0, len(seg_audio))]

    if r is None or ('onsets' not in r) or ('offsets' not in r):
        return [(0, len(seg_audio))]

    on = np.array(r['onsets']).astype(float)
    off = np.array(r['offsets']).astype(float)
    if len(on) == 0 or len(off) == 0:
        return [(0, len(seg_audio))]

    dur_s = len(seg_audio) / SR
    as_seconds = float(np.nanmax(off)) <= (dur_s * 2.5)

    out = []
    for a, b in zip(on, off):
        if as_seconds:
            s0 = int(a * SR)
            s1 = int(b * SR)
        else:
            s0 = int(a)
            s1 = int(b)
        s0 = max(0, min(s0, len(seg_audio)))
        s1 = max(0, min(s1, len(seg_audio)))
        if s1 - s0 >= 100:
            out.append((s0, s1))

    return out if out else [(0, len(seg_audio))]


def flush_batch(batch_meta, batch_mel):
    X = np.stack(batch_mel).astype(np.float32)
    rows = [
        {'FileID': fid_, 'FileName': fn, 'StartSample': st, 'Context': ctx, 'Emitter': em}
        for fid_, fn, st, ctx, em in batch_meta
    ]
    return rows, X


segment_rows = []
mel_chunks = []
batch_meta = []
batch_mel = []

files_loaded = 0
skip_missing_fileinfo = 0
skip_missing_wav = 0
skip_read_error = 0
skip_timeout = 0
skip_too_short = 0

for fid, grp in tqdm(annotations.groupby('FileID'), desc='WAV -> dynamic subseg -> mel'):
    if MAX_SEGMENTS is not None and (len(segment_rows) + len(batch_mel)) >= MAX_SEGMENTS:
        break
    if MAX_FILES is not None and files_loaded >= MAX_FILES:
        break

    if fid not in file_info.index:
        skip_missing_fileinfo += 1
        continue

    wav_path = file_info.loc[fid, 'wav_path']
    if not Path(wav_path).exists():
        skip_missing_wav += 1
        continue

    try:
        with ThreadPoolExecutor(max_workers=1) as ex:
            fut = ex.submit(_load_wav, wav_path)
            y, _ = fut.result(timeout=WAV_LOAD_TIMEOUT_SEC)
    except FuturesTimeoutError:
        skip_timeout += 1
        continue
    except Exception:
        skip_read_error += 1
        continue

    fname = file_info.loc[fid, 'FileName']
    added_here = 0

    for _, row in grp.iterrows():
        if MAX_SEGMENTS is not None and (len(segment_rows) + len(batch_mel)) >= MAX_SEGMENTS:
            break

        ann_start = int(row['Start sample'])
        ann_end = int(row['End sample'])
        seg = y[ann_start:ann_end].astype(np.float32)
        if len(seg) < 100:
            continue

        sub_ranges = dynamic_subsegments(seg)
        for s0, s1 in sub_ranges:
            sub = seg[s0:s1]
            if len(sub) < 100:
                continue
            batch_meta.append((
                int(fid),
                fname,
                ann_start + int(s0),
                int(row['Context']),
                int(row['Emitter']),
            ))
            batch_mel.append(make_mel(sub))
            added_here += 1

        if len(batch_mel) >= BATCH_TRANSFORM_SEGMENTS:
            rows, X = flush_batch(batch_meta, batch_mel)
            segment_rows.extend(rows)
            mel_chunks.append(X)
            batch_meta, batch_mel = [], []

    del y

    if added_here == 0:
        skip_too_short += 1
    else:
        files_loaded += 1

# final flush
if len(batch_mel) > 0:
    rows, X = flush_batch(batch_meta, batch_mel)
    segment_rows.extend(rows)
    mel_chunks.append(X)
    batch_meta, batch_mel = [], []

seg_df = pd.DataFrame(segment_rows)
mel_matrix = np.vstack(mel_chunks) if len(mel_chunks) else np.empty((0, N_MELS * PAD_LEN), dtype=np.float32)

if len(seg_df) != len(mel_matrix):
    raise RuntimeError(f'Mismatch seg_df={len(seg_df)} mel_matrix={len(mel_matrix)}')

print('Segments:', len(seg_df), '| mel_matrix shape:', mel_matrix.shape)
print('Files loaded:', files_loaded)
print('Skipped: missing fileinfo=', skip_missing_fileinfo,
      '| missing wav=', skip_missing_wav,
      '| read error=', skip_read_error,
      '| timeout=', skip_timeout,
      '| too short=', skip_too_short)


## 4) Top-5 emitter model selection (Silhouette + ARI/NMI proxy) and full clustering


In [ ]:
import umap
import hdbscan
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.cluster import AgglomerativeClustering
import librosa

if len(seg_df) == 0:
    raise RuntimeError('No segments were created. Check WAV paths / segmentation settings.')

# ---- model selection on top-5 emitters ----
MODELSEL_MAX_PER_EMITTER = 3000
PROXY_MAX_PER_EMITTER = 60

emit_counts = seg_df['Emitter'].value_counts()
top_emitters = emit_counts.head(5).index.tolist()
print('Top-5 emitters:', top_emitters)

rng = np.random.default_rng(RANDOM_STATE)
subset_idx = []
for em in top_emitters:
    ix = np.where(seg_df['Emitter'].values == em)[0]
    if len(ix) > MODELSEL_MAX_PER_EMITTER:
        ix = rng.choice(ix, size=MODELSEL_MAX_PER_EMITTER, replace=False)
    subset_idx.extend(ix.tolist())
subset_idx = np.array(sorted(subset_idx), dtype=int)

mel_subset = mel_matrix[subset_idx]
seg_subset = seg_df.iloc[subset_idx].reset_index(drop=True)

print('Model selection subset size:', len(seg_subset))

# Build DTW-MFCC proxy labels (per emitter), like paper idea
proxy_labels = np.full(len(seg_subset), -99, dtype=int)  # -99 means not included in proxy scoring
next_label_offset = 0

for em in top_emitters:
    ix_em = np.where(seg_subset['Emitter'].values == em)[0]
    if len(ix_em) == 0:
        continue
    if len(ix_em) > PROXY_MAX_PER_EMITTER:
        ix_em = rng.choice(ix_em, size=PROXY_MAX_PER_EMITTER, replace=False)

    specs = mel_subset[ix_em].reshape(len(ix_em), N_MELS, PAD_LEN)

    # derive MFCC from log-mel for DTW proxy
    mfccs = [librosa.feature.mfcc(S=s, n_mfcc=13) for s in specs]
    n = len(mfccs)
    if n < 3:
        continue

    dist = np.zeros((n, n), dtype=np.float32)
    for i in range(n):
        for j in range(i + 1, n):
            D, wp = librosa.sequence.dtw(X=mfccs[i], Y=mfccs[j], metric='euclidean')
            d = float(D[-1, -1]) / max(len(wp), 1)
            dist[i, j] = d
            dist[j, i] = d

    tri = dist[np.triu_indices(n, 1)]
    if len(tri) == 0:
        continue
    thr = float(np.quantile(tri, 0.05))

    ac = AgglomerativeClustering(
        n_clusters=None,
        metric='precomputed',
        linkage='average',
        distance_threshold=thr,
    )
    lbl = ac.fit_predict(dist).astype(int)

    proxy_labels[ix_em] = lbl + next_label_offset
    next_label_offset += int(lbl.max()) + 1

proxy_mask = proxy_labels != -99
print('Proxy-labeled points for ARI/NMI:', int(proxy_mask.sum()))

# Candidate manifold/clustering settings
UMAP_CANDIDATES = [
    (30, 0.3, 'euclidean'),
    (30, 0.0, 'euclidean'),
    (15, 0.0, 'euclidean'),
]

trials = []
best = None

for n_nb, min_dist, metric in UMAP_CANDIDATES:
    reducer = umap.UMAP(
        n_components=2,
        n_neighbors=n_nb,
        min_dist=min_dist,
        metric=metric,
        random_state=RANDOM_STATE,
    )
    emb_sub = reducer.fit_transform(mel_subset).astype(np.float32)

    for frac in HDBSCAN_FRAC_CANDIDATES:
        mcs = max(10, int(len(emb_sub) * frac))
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=mcs,
            min_samples=HDBSCAN_MIN_SAMPLES,
            cluster_selection_epsilon=HDBSCAN_EPS,
            cluster_selection_method=HDBSCAN_METHOD,
            metric='euclidean',
            prediction_data=True,
        )
        labels_sub = clusterer.fit_predict(emb_sub).astype(int)
        n_clusters = len(set(labels_sub)) - (1 if -1 in labels_sub else 0)

        non_noise = labels_sub >= 0
        sil = -1.0
        if non_noise.sum() > 20 and n_clusters > 1:
            try:
                sil = float(silhouette_score(emb_sub[non_noise], labels_sub[non_noise]))
            except Exception:
                sil = -1.0

        ari = np.nan
        nmi = np.nan
        if proxy_mask.sum() > 10:
            try:
                ari = float(adjusted_rand_score(proxy_labels[proxy_mask], labels_sub[proxy_mask]))
                nmi = float(normalized_mutual_info_score(proxy_labels[proxy_mask], labels_sub[proxy_mask]))
            except Exception:
                ari = np.nan
                nmi = np.nan

        # paper-like objective: internal consistency + agreement with acoustic proxy
        score = sil
        if not np.isnan(ari):
            score += 0.5 * ari
        if not np.isnan(nmi):
            score += 0.5 * nmi

        row = {
            'n_neighbors': n_nb,
            'min_dist': min_dist,
            'metric': metric,
            'frac': frac,
            'min_cluster_size': mcs,
            'n_clusters_sub': n_clusters,
            'silhouette_sub': sil,
            'ari_proxy': ari,
            'nmi_proxy': nmi,
            'score': score,
        }
        trials.append(row)

        if best is None or score > best['score']:
            best = row.copy()

trial_df = pd.DataFrame(trials).sort_values('score', ascending=False)
print('Top model-selection trials:')
print(trial_df.head(10))
print('Selected config:', best)

# ---- full fit on all segments with selected config ----
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=int(best['n_neighbors']),
    min_dist=float(best['min_dist']),
    metric=best['metric'],
    random_state=RANDOM_STATE,
)
embedding_all = reducer.fit_transform(mel_matrix).astype(np.float32)

mcs_full = max(10, int(len(embedding_all) * float(best['frac'])))
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=mcs_full,
    min_samples=HDBSCAN_MIN_SAMPLES,
    cluster_selection_epsilon=HDBSCAN_EPS,
    cluster_selection_method=HDBSCAN_METHOD,
    metric='euclidean',
    prediction_data=True,
)
labels_full = clusterer.fit_predict(embedding_all).astype(int)
seg_df['syllable_id'] = labels_full

n_clusters = len(set(labels_full)) - (1 if -1 in labels_full else 0)
n_noise = int((labels_full == -1).sum())
print('Full-data clustering:')
print({'n_clusters': n_clusters, 'noise': n_noise, 'total': len(labels_full), 'min_cluster_size': mcs_full})


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 5))

# context distribution
ctx = seg_df['Context'].value_counts().sort_index()
axes[0].bar([CONTEXT_NAMES.get(c, str(c)) for c in ctx.index], ctx.values, color='steelblue', edgecolor='black', linewidth=0.5)
axes[0].set_title('Segments per context')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=35, labelsize=8)

# subsample for plotting speed
max_plot = 100_000
if len(seg_df) > max_plot:
    idx = np.random.default_rng(RANDOM_STATE).choice(len(seg_df), max_plot, replace=False)
else:
    idx = np.arange(len(seg_df))

emb = embedding_all[idx]
sp = seg_df.iloc[idx]

sc1 = axes[1].scatter(emb[:,0], emb[:,1], c=sp['Context'].astype(int), s=3, alpha=0.45, cmap='tab10')
axes[1].set_title('UMAP by Context')
axes[1].set_xlabel('UMAP 1'); axes[1].set_ylabel('UMAP 2')
plt.colorbar(sc1, ax=axes[1], label='Context')

lab = sp['syllable_id'].values
noise = lab == -1
axes[2].scatter(emb[noise,0], emb[noise,1], c='lightgray', s=2, alpha=0.25, label='noise')
sc2 = axes[2].scatter(emb[~noise,0], emb[~noise,1], c=lab[~noise], s=3, alpha=0.65, cmap='tab20')
axes[2].set_title('UMAP by HDBSCAN syllable')
axes[2].set_xlabel('UMAP 1'); axes[2].set_ylabel('UMAP 2')
plt.colorbar(sc2, ax=axes[2], label='Syllable ID')

plt.tight_layout()
plt.show()

print('Distinct syllable labels (incl noise):', seg_df['syllable_id'].nunique())
print('Distinct syllable labels (excl noise):', seg_df[seg_df['syllable_id'] >= 0]['syllable_id'].nunique())


## 5) Build per-file sequences (paper filter >=30 per context)


In [ ]:
by_file = (
    seg_df.sort_values(['FileName', 'StartSample'])
    .groupby('FileName', sort=False)
    .agg(
        seq=('syllable_id', list),
        context=('Context', lambda x: int(x.mode().iloc[0])),
        emitter=('Emitter', 'first'),
    )
    .reset_index()
)

by_file['seq'] = by_file['seq'].apply(lambda s: [x for x in s if x >= 0])
by_file = by_file[by_file['seq'].apply(len) >= 2].copy()

ctx_counts = by_file['context'].value_counts()
keep_ctx = ctx_counts[ctx_counts >= MIN_SAMPLES_PER_CONTEXT].index
by_file = by_file[by_file['context'].isin(keep_ctx)].copy()

print('Files with >=2 non-noise syllables:', len(by_file))
print('Contexts kept >=%d:' % MIN_SAMPLES_PER_CONTEXT, sorted(by_file['context'].unique().tolist()))
print(by_file['context'].map(CONTEXT_NAMES).value_counts().sort_index())


## 6) HP1: sequence features (a-r) + SVM + permutation


In [ ]:
import math
import collections
import networkx as nx
from itertools import pairwise

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.svm import SVC
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay


def num_transition_types_in_context(df, context_id):
    pairs = set()
    for seq in df[df['context'] == context_id]['seq']:
        pairs.update(pairwise(seq))
    return len(pairs)


def prob_syl_by_context(df, context_id):
    freq = df[df['context'] == context_id]['seq'].explode().value_counts()
    total = freq.sum()
    return (freq / total).to_dict() if total else {}


def compute_conditional_prob_1(series, n):
    from collections import defaultdict
    cond_freqs = defaultdict(lambda: defaultdict(int))
    for seq in series:
        ants = []
        for i in range(n):
            if i < len(seq):
                ants.append(seq[i])
        for item in seq:
            cond_freqs[tuple(ants)][item] += 1
            if len(ants) >= n:
                ants.pop(0)
            ants.append(item)
    out = {}
    for key, d in cond_freqs.items():
        s = sum(d.values())
        out[key] = {k: v / s for k, v in d.items()}
    return out


def transitions_dict(sequences):
    transitions = sequences.apply(lambda x: [p for p in pairwise(x)]).explode().value_counts()
    total = transitions.sum()
    return transitions.apply(lambda x: x / total).to_dict() if total else {}


def compute_transition_prob(series):
    g = nx.DiGraph()
    for seq in series:
        for i in range(len(seq) - 1):
            a, b = seq[i], seq[i + 1]
            if g.has_edge(a, b):
                g[a][b]['weight'] += 1
            else:
                g.add_edge(a, b, weight=1)
    trans_probs = {}
    for source in g.nodes():
        total_weight = sum(g[source][target]['weight'] for target in g.successors(source))
        trans_probs[source] = {target: g[source][target]['weight'] / total_weight for target in g.successors(source)}
    return trans_probs


def make_graph_transitions(df):
    G = nx.DiGraph()
    seq_series = df['seq']
    total_syllables = len(seq_series.explode().unique())

    for n, f in seq_series.explode().value_counts().to_dict().items():
        G.add_node(n, frequency=f, p_frequency=f / max(total_syllables, 1))

    edges = [p for seq in seq_series for p in pairwise(seq)]
    for e, v in collections.Counter(edges).items():
        current = e[0]
        posterior = [x for x in edges if x[0] == current]
        p_trans = (v / len(posterior)) if posterior else 1e-7
        target = e[1]
        antecedent = [x for x in edges if x[1] == target]
        p_cond = (v / len(antecedent)) if antecedent else 1e-7
        G.add_edge(*e, frequency=v, p_trans=p_trans, p_cond=p_cond)

    return G


def prepare_data_from_sequences(df, data_df, G):
    columns = list(map(chr, range(97, 97 + 18)))  # a-r
    out = {c: [] for c in columns}

    contexts = sorted(df['context'].unique())
    _transitions_in_context = {c: num_transition_types_in_context(df, c) for c in contexts}
    _prob_syl_by_context = {c: prob_syl_by_context(df, c) for c in contexts}
    _cond_prob_by_context = {c: compute_conditional_prob_1(df[df['context'] == c]['seq'], 1) for c in contexts}
    _trans_probs = {c: compute_transition_prob(df[df['context'] == c]['seq']) for c in contexts}
    _transitions_dict = {c: transitions_dict(df[df['context'] == c]['seq']) for c in contexts}

    _cond_prob_all = compute_conditional_prob_1(df['seq'], 1)
    _cond_prob_all_2 = compute_conditional_prob_1(df['seq'], 2)
    _transitions_dict_all = transitions_dict(df['seq'])
    _trans_probs_all = compute_transition_prob(df['seq'])
    _transitions_in_total = max(1, int(df['seq'].apply(lambda x: [p for p in pairwise(x)]).explode().value_counts().sum()))
    freq_syl = df['seq'].explode().value_counts()
    total_freq = max(1, int(freq_syl.sum()))
    _prob_syl_in_total = (freq_syl / total_freq).to_dict()

    for _, row in data_df.iterrows():
        seq = row['seq']
        c = row['context']

        a = len(set(seq))
        b = len(seq)
        c_trans = len(list(pairwise(seq)))
        d = a / max(c_trans, 1)

        if c in _transitions_in_context:
            e = len(set(pairwise(seq))) / max(_transitions_in_context[c], 1)
            p_ctx = _prob_syl_by_context[c]
            f = -sum([p_ctx.get(s, 1e-9) * np.log2(max(p_ctx.get(s, 1e-9), 1e-9)) for s in seq])
            init_prob = p_ctx
            cond1 = _cond_prob_by_context[c]
            trans_dict = _transitions_dict[c]
            trans_prob = _trans_probs[c]
        else:
            e = len(set(pairwise(seq))) / _transitions_in_total
            f = -sum([_prob_syl_in_total.get(s, 1e-9) * np.log2(max(_prob_syl_in_total.get(s, 1e-9), 1e-9)) for s in seq])
            init_prob = _prob_syl_in_total
            cond1 = _cond_prob_all
            trans_dict = _transitions_dict_all
            trans_prob = _trans_probs_all

        g_val = init_prob.get(seq[0], 1e-9)
        for i in range(1, len(seq)):
            ant, cur = seq[i - 1], seq[i]
            g_val *= cond1.get((ant,), {}).get(cur, 1e-9)

        h = np.prod([trans_dict.get(p, 1e-9) for p in pairwise(seq)]) if len(seq) > 1 else 0
        i_val = a / max(b, 1)

        j = 0.0
        for p in pairwise(seq):
            tr = trans_prob.get(p[0], {}).get(p[1], 1e-9)
            j += tr * np.log2(max(tr, 1e-9))
        j = -j

        probs_k = [G.edges[p]['p_trans'] for p in pairwise(seq) if p in G.edges]
        k = np.prod(probs_k) if probs_k else 0

        probs_l = [G.edges[p]['p_cond'] * np.log2(max(G.edges[p]['p_cond'], 1e-9)) for p in pairwise(seq) if p in G.edges]
        l = np.prod(probs_l) if probs_l else 0

        probs_m = [G.edges[p]['p_trans'] * np.log2(max(G.edges[p]['p_trans'], 1e-9)) for p in pairwise(seq) if p in G.edges]
        m = np.prod(probs_m) if probs_m else 0

        p_cond_n = [init_prob.get(seq[0], 1e-9)]
        for i2 in range(1, len(seq)):
            ant, cur = seq[i2 - 1], seq[i2]
            p_cond_n.append(_cond_prob_all.get((ant,), {}).get(cur, 1e-9))
        n = np.prod(p_cond_n) if p_cond_n else 0

        p_cond_o = [G.nodes[seq[0]]['p_frequency'] if seq[0] in G.nodes else 1e-9]
        p_trans_p = []
        for i2 in range(1, len(seq)):
            ant, cur = seq[i2 - 1], seq[i2]
            p_cond_o.append(G[ant][cur]['p_cond'] if (ant in G and cur in G[ant]) else 1e-9)
            if i2 < len(seq) - 1:
                suc = seq[i2 + 1]
                p_trans_p.append(G[cur][suc]['p_trans'] if (cur in G and suc in G[cur]) else 1e-9)
        o = np.prod(p_cond_o) if p_cond_o else 0
        pval = np.prod(p_trans_p) if p_trans_p else 0

        p_cond_q = [init_prob.get(seq[0], 1e-9)]
        for i2 in range(1, len(seq)):
            ant, cur = seq[i2 - 1], seq[i2]
            v = _cond_prob_all.get((ant,), {}).get(cur, 1e-9)
            if i2 > 1:
                ant2 = seq[i2 - 2]
                v *= _cond_prob_all_2.get((ant2, ant), {}).get(cur, 1e-9)
            p_cond_q.append(v)
        q = np.prod(p_cond_q) if p_cond_q else 0
        r = np.power(np.prod([1 / max(x, 1e-9) for x in p_cond_q]), 1 / max(len(p_cond_q), 1)) if p_cond_q else 0

        vals = [a, b, c_trans, d, e, f, g_val, h, i_val, j, k, l, m, n, o, pval, q, r]
        for col, val in zip(columns, vals):
            out[col].append(val)

    return pd.DataFrame(out)


if len(by_file) < 20:
    print('Too few files after filter for HP1. Need more data.')
else:
    G = make_graph_transitions(by_file)
    feat = prepare_data_from_sequences(by_file, by_file, G)
    feat = feat.replace([np.inf, -np.inf], np.nan).fillna(0)

    y = by_file['context'].values
    le = LabelEncoder()
    y_enc = le.fit_transform(y)

    X_train, X_test, y_train, y_test = train_test_split(feat, y_enc, test_size=0.2, stratify=y_enc, random_state=RANDOM_STATE)

    sc = StandardScaler()
    X_train_std = sc.fit_transform(X_train)
    X_test_std = sc.transform(X_test)

    cv_folds = max(2, min(10, int(np.bincount(y_train).min())))
    cv = StratifiedKFold(n_splits=cv_folds, shuffle=True, random_state=RANDOM_STATE)

    param_grid = {'C': np.logspace(-3, 3, 7), 'gamma': np.logspace(-3, 3, 7), 'kernel': ['rbf']}
    grid = GridSearchCV(SVC(), param_grid, cv=cv, scoring='f1_weighted', n_jobs=-1)
    grid.fit(X_train_std, y_train)

    y_pred = grid.best_estimator_.predict(X_test_std)
    f1_orig = f1_score(y_test, y_pred, average='weighted')

    # Permutation
    by_file_perm = by_file.copy()
    by_file_perm['seq'] = by_file_perm['seq'].apply(lambda x: list(np.random.permutation(x)))
    feat_perm = prepare_data_from_sequences(by_file, by_file_perm, G).replace([np.inf, -np.inf], np.nan).fillna(0)
    X_perm_std = sc.transform(feat_perm)
    y_perm = le.transform(by_file_perm['context'].values)
    y_perm_pred = grid.best_estimator_.predict(X_perm_std)
    f1_perm = f1_score(y_perm, y_perm_pred, average='weighted')

    print('HP1 results')
    print('SVM best params:', grid.best_params_)
    print('F1 weighted (original):', round(f1_orig, 4))
    print('F1 weighted (permuted):', round(f1_perm, 4))
    print('delta:', round(abs(f1_orig - f1_perm), 4))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=axes[0], colorbar=False)
    axes[0].set_title('Original')
    ConfusionMatrixDisplay.from_predictions(y_perm, y_perm_pred, ax=axes[1], colorbar=False)
    axes[1].set_title('Permuted')
    plt.tight_layout(); plt.show()


## 7) HP2: Wilcoxon rank-sum on syllable usage


In [ ]:
from scipy.stats import ranksums
import itertools

if len(by_file) == 0:
    print('No by_file data.')
else:
    all_syl = sorted(seg_df[seg_df['syllable_id'] >= 0]['syllable_id'].unique().tolist())
    contexts = sorted(by_file['context'].unique().tolist())

    # heatmap
    freq_cols = {}
    for c in contexts:
        vc = by_file[by_file['context'] == c]['seq'].explode().value_counts().reindex(all_syl, fill_value=0)
        freq_cols[c] = vc / max(vc.sum(), 1)
    heat = pd.DataFrame(freq_cols)

    plt.figure(figsize=(11, 5))
    sns.heatmap(heat, cmap='YlOrRd')
    plt.title('Syllable usage by context')
    plt.xticks(np.arange(len(contexts)) + 0.5, [CONTEXT_NAMES.get(c, str(c)) for c in contexts], rotation=35, ha='right')
    plt.tight_layout(); plt.show()

    print('Pairwise Wilcoxon rank-sum (p<0.05 significant):')
    for c1, c2 in itertools.combinations(contexts, 2):
        x = by_file[by_file['context'] == c1]['seq'].explode().value_counts().reindex(all_syl, fill_value=0)
        y = by_file[by_file['context'] == c2]['seq'].explode().value_counts().reindex(all_syl, fill_value=0)
        _, p = ranksums(x, y)
        print(f'{CONTEXT_NAMES.get(c1, c1):20s} vs {CONTEXT_NAMES.get(c2, c2):20s}  p={p:.4g}')


## 8) HP3: Maximal Repeats + power-law vs exponential


In [ ]:
try:
    from suffix_tree import Tree
    HAS_SUFFIX_TREE = True
except Exception:
    HAS_SUFFIX_TREE = False
    print('Install suffix_tree: pip install suffix_tree')

if HAS_SUFFIX_TREE and len(by_file) > 0:
    seqs = {i: s for i, s in enumerate(by_file['seq'].tolist())}
    tree = Tree(seqs)

    mr_lengths = []
    for C, path in tree.maximal_repeats():
        toks = str(path).split()
        try:
            vals = [int(t) for t in toks if int(t) >= 0]
        except Exception:
            vals = []
        if len(vals) >= 2:
            mr_lengths.append(len(vals))

    mr_lengths = pd.Series(mr_lengths)
    print('MR count:', len(mr_lengths))
    print(mr_lengths.describe())

    plt.figure(figsize=(8, 4))
    plt.hist(mr_lengths, bins=40, color='purple', edgecolor='black', alpha=0.8)
    plt.title('Maximal Repeat lengths')
    plt.xlabel('Length'); plt.ylabel('Count')
    plt.tight_layout(); plt.show()

    try:
        import powerlaw
        data = mr_lengths[mr_lengths > 1].tolist()
        if len(data) > 30:
            fit = powerlaw.Fit(data, discrete=True, verbose=False)
            R, p = fit.distribution_compare('power_law', 'exponential', normalized_ratio=True)
            print('alpha=', round(fit.alpha, 4), 'sigma=', round(fit.sigma, 4), 'xmin=', fit.xmin)
            print('LR test power_law vs exponential: R=', round(R, 4), 'p=', round(p, 6))
        else:
            print('Not enough MR samples for powerlaw test')
    except Exception:
        print('Install powerlaw: pip install powerlaw')


## 9) Context transition network metrics


In [ ]:
import networkx as nx
from itertools import pairwise


def build_ctx_graph(df, ctx):
    g = nx.DiGraph()
    seqs = df[df['context'] == ctx]['seq']
    for seq in seqs:
        for a, b in pairwise(seq):
            if g.has_edge(a, b):
                g[a][b]['weight'] += 1
            else:
                g.add_edge(a, b, weight=1)
    return g


def smallworld_proxy(g):
    gu = g.to_undirected()
    if gu.number_of_nodes() < 4 or gu.number_of_edges() < 2:
        return np.nan, np.nan
    c = nx.average_clustering(gu)
    er = nx.erdos_renyi_graph(gu.number_of_nodes(), nx.density(gu), seed=RANDOM_STATE)
    c_rand = nx.average_clustering(er) if er.number_of_edges() > 0 else np.nan
    omega = (c_rand / c) if (c > 0 and not np.isnan(c_rand)) else np.nan
    return c, omega

if len(by_file) > 0:
    contexts = sorted(by_file['context'].unique().tolist())
    print(f"{'Context':20s} {'Nodes':>6s} {'Edges':>6s} {'Density':>8s} {'AvgC':>8s} {'omega*':>8s}")
    print('-' * 65)
    for c in contexts:
        g = build_ctx_graph(by_file, c)
        d = nx.density(g) if g.number_of_nodes() > 1 else np.nan
        avg_c, om = smallworld_proxy(g)
        print(f"{CONTEXT_NAMES.get(c, str(c)):20s} {g.number_of_nodes():6d} {g.number_of_edges():6d} {d:8.4f} {avg_c:8.4f} {om:8.4f}")


## Notes

- If you still see only ~2 coarse groups, verify that `vocalseg` is installed and `USE_DYNAMIC_SUBSEG=True`.
- The refined result in Assom uses both dynamic sub-segmentation and tuned manifold/clustering hyperparameters.
- For fast debugging use `MAX_FILES` / `MAX_SEGMENTS`; for final reproduction set both to `None`.
